# Predicción de retrasos en vuelos comerciales
Grupo 11: Mikel Lorite, Adrián Izquierdo y Jon Alba

## Contexto y objetivos del proyecto
El contexto de este proyecto se enmarca en la industria de la aviación comercial, un sector que genera cantidades masivas de datos diariamente y donde la puntualidad operativa es un factor crítico. Para llevar a cabo este estudio, utilizamos el conjunto de datos "2015 Flight Delays and Cancellations", publicado originalmente por la Oficina de Estadísticas de Transporte (Bureau of Transportation Statistics) del Departamento de Transporte de los Estados Unidos (DOT). Este dataset rastrea el rendimiento y la puntualidad de los vuelos domésticos operados por las grandes aerolíneas comerciales dentro del país durante el año 2015.

El objetivo principal de nuestro trabajo es construir un modelo predictivo escalable utilizando Apache Spark que permita anticipar si un vuelo sufrirá un retraso significativo a su llegada. A través de este análisis, buscamos identificar patrones subyacentes en las demoras, respondiendo a preguntas sobre qué factores —como la aerolínea, las infraestructuras de origen/destino o las franjas horarias— inciden en la probabilidad de que un vuelo no cumpla con su horario programado. Adicionalmente, desde la perspectiva de la Ingeniería de Datos, el proyecto tiene como meta evaluar la eficiencia computacional de diversos algoritmos de clasificación binaria (Regresión Logística, Random Forest y Gradient-Boosted Trees) frente a un escenario de alto volumen de datos, midiendo y documentando empíricamente la escalabilidad del clúster a través de pruebas de speed-up y size-up.

## Descripción de los datos 
El conjunto de datos seleccionado representa un volumen de información de gran magnitud, ideal para el procesamiento distribuido. En su totalidad, consta de más de 5,8 millones de registros (5.819.079 observaciones empíricas) estructurados originalmente en 31 variables. Para garantizar la integridad relacional de la información, el dataset se divide en tres archivos en formato CSV independientes pero interconectados:

* flights.csv: Constituye el núcleo central de la información, recogiendo el registro individual de cada vuelo operado durante el año. Entre sus variables métricas y categóricas se incluye información temporal exhaustiva (fecha, horarios de salida y llegada programados frente a los reales) e indicadores de rendimiento operativo. La variable clave para nuestro problema de clasificación subyace en la columna de retraso a la llegada (Arrival Delay), que nos indica en minutos la diferencia respecto al horario previsto.

* airlines.csv: Funciona como una tabla de dimensión o diccionario. Contiene el mapeo directo entre los identificadores de código IATA y el nombre comercial estandarizado de cada compañía aérea, lo que resulta fundamental para la posterior visualización e interpretación de los modelos.

* airports.csv: Actúa como un catálogo geográfico detallado. Este archivo permite enriquecer los datos de los vuelos enlazando los códigos IATA de los aeropuertos de origen y destino con su ubicación física. Proporciona variables demográficas y geoespaciales clave como la ciudad, el estado, la latitud y la longitud, abriendo la puerta a capturar el impacto de la congestión regional en nuestras predicciones.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os


spark = SparkSession.builder.appName("FlightDelays").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/01 17:34:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
import pyspark.sql.functions as F

df_flights = spark.read.csv("data/flights.csv", header=True, inferSchema=True)
df_airlines = spark.read.csv("data/airlines.csv", header=True, inferSchema=True)
df_airports = spark.read.csv("data/airports.csv", header=True, inferSchema=True)

print(f"Total de vuelos iniciales: {df_flights.count():,}")

Total de vuelos iniciales: 5,819,079
root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = tr

In [ ]:
df_joined = df_flights.join(
    F.broadcast(df_airlines), 
    df_flights.AIRLINE == df_airlines.IATA_CODE, 
    "left"
).drop("IATA_CODE") 

df_airports_orig = df_airports.select(
    F.col("IATA_CODE").alias("ORIGIN_IATA"),
    F.col("LATITUDE").alias("ORIGIN_LAT"),
    F.col("LONGITUDE").alias("ORIGIN_LONG")
)

df_joined = df_joined.join(
    F.broadcast(df_airports_orig), 
    df_joined.ORIGIN_AIRPORT == df_airports_orig.ORIGIN_IATA, 
    "left"
).drop("ORIGIN_IATA")

# Puedes repetir este último paso para DESTINATION_AIRPORT si lo consideras útil

In [5]:
# Filtrar cancelados y desviados
df_clean = df_joined.filter((col("CANCELLED") == 0) & (col("DIVERTED") == 0))

# Contar nulos en las columnas clave
df_clean.select([count(when(isnan(c) | col(c).isNull(), c)).alias(c) for c in ["ARRIVAL_DELAY", "DEPARTURE_DELAY"]]).show()

NameError: name 'df_joined' is not defined